# LOAD DATA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/MRIarchieve.zip'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.printdir()  # prints all files inside the ZIP


# HANDCRAFTED FEATURE EXTRACTION

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import local_binary_pattern
from skimage import measure

# Path to extracted dataset
base_path = '/content/MRIarchieve'

# LBP parameters
radius = 1
n_points = 8 * radius

def extract_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # Resize to standard size
    img = cv2.resize(img, (128, 128))

    features = []

    # Intensity features
    features.extend([np.mean(img), np.std(img), np.max(img), np.min(img), np.median(img)])

    # Texture features: LBP histogram
    lbp = local_binary_pattern(img, n_points, radius, method='uniform')
    n_bins = int(lbp.max() + 1)
    hist, _ = np.histogram(lbp, bins=n_bins, range=(0, n_bins), density=True)
    features.extend(hist.tolist())

    # Shape features using simple thresholding
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    labels = measure.label(mask)
    props = measure.regionprops(labels)
    if props:
        largest = max(props, key=lambda x: x.area)
        features.extend([largest.area, largest.perimeter, largest.eccentricity, largest.solidity])
    else:
        features.extend([0,0,0,0])

    return features

# Map class names to numeric labels
class_mapping = {}
label_counter = 0

def process_folder(folder_path):
    global label_counter
    X = []
    y = []

    for class_folder in os.listdir(folder_path):
        class_path = os.path.join(folder_path, class_folder)
        if not os.path.isdir(class_path):
            continue

        if class_folder not in class_mapping:
            class_mapping[class_folder] = label_counter
            label_counter += 1

        for img_file in os.listdir(class_path):
            img_path = os.path.join(class_path, img_file)
            feats = extract_features(img_path)
            if feats:
                X.append(feats)
                y.append(class_mapping[class_folder])

    return X, y

# Process Training and Testing folders
train_path = os.path.join(base_path, 'Training')
test_path = os.path.join(base_path, 'Testing')

X_train, y_train = process_folder(train_path)
X_test, y_test = process_folder(test_path)

# Convert to DataFrame
feature_names = ['mean','std','max','min','median']
lbp_cols = [f'lbp_{i}' for i in range(int(n_points+2))]
shape_cols = ['area','perimeter','eccentricity','solidity']

df_train = pd.DataFrame(X_train, columns=feature_names + lbp_cols + shape_cols)
df_train['label'] = y_train

df_test = pd.DataFrame(X_test, columns=feature_names + lbp_cols + shape_cols)
df_test['label'] = y_test

# Save CSVs
train_csv = '/content/drive/MyDrive/Handcrafted_Train_features_multiclass.csv'
test_csv = '/content/drive/MyDrive/Handcrafted_Test_features_multiclass.csv'

df_train.to_csv(train_csv, index=False)
df_test.to_csv(test_csv, index=False)

print("Training features saved to:", train_csv)
print("Testing features saved to:", test_csv)
print("Class mapping:", class_mapping)
print("Training rows:", df_train.shape[0], "Testing rows:", df_test.shape[0])
